In [ ]:
import os
import pandas as pd
# todo

In [2]:
def merge_usage_dir(input_dir: str, output_file: str) -> None:
    """
    合并一个 usage 目录下的所有 csv 文件，并输出为一个 all.csv

    逻辑：
    1. 递归读取目录下所有 csv
    2. 将 sample 列中的 '__' 后缀去掉，只保留前半部分
    3. 按共同列 outer merge
    """
    file_paths = []
    for dirname, _, filenames in os.walk(input_dir):
        for filename in filenames:
            if filename.endswith(".csv"):
                file_paths.append(os.path.join(dirname, filename))

    if not file_paths:
        print(f"[跳过] 目录下没有 csv 文件: {input_dir}")
        return

    file_paths = sorted(file_paths)
    print(f"[开始] {input_dir}，共找到 {len(file_paths)} 个 csv 文件")

    df_all = None

    for file_path in file_paths:
        print(f"  读取: {file_path}")
        df = pd.read_csv(file_path)

        if "sample" not in df.columns:
            print(f"  [跳过] 文件中没有 sample 列: {file_path}")
            continue

        # 标准化 sample，只保留 '__' 前面的部分
        values = [
            str(sample_name).split("__")[0] for sample_name in df["sample"].tolist()
        ]
        df = df.drop(columns=["sample"])
        df.insert(loc=0, column="sample", value=values)

        if df_all is None:
            df_all = df.copy()
        else:
            df_all = pd.merge(df_all, df, how="outer")

    if df_all is None:
        print(f"[跳过] 没有可合并的数据: {input_dir}")
        return

    df_all.to_csv(os.path.join(input_dir, output_file), index=False)
    print(f"[完成] 输出文件: {output_file}")
    print(df_all.head())

In [3]:
base_dir = "."

In [4]:
target_dirs = ["1Vusage", "1Jusage", "1VJusage"]

for dir_name in target_dirs:
    input_dir = os.path.join(base_dir, dir_name)
    output_file = os.path.join(base_dir, f"df_{dir_name}_all.csv")

    if os.path.isdir(input_dir):
        merge_usage_dir(input_dir, output_file)
    else:
        print(f"[跳过] 目录不存在: {input_dir}")

[跳过] 目录不存在: ./1Vusage
[跳过] 目录不存在: ./1Jusage
[开始] ./1VJusage，共找到 2 个 csv 文件
  读取: ./1VJusage/TRA.csv
  读取: ./1VJusage/TRB.csv
[完成] 输出文件: ./df_1VJusage_all.csv
           sample      Category  TRAV1-1;TRAJ1  TRAV1-1;TRAJ10  \
0  SF_11_808012_2  experimental            NaN        0.000045   
1    SF_11_4005_1  experimental            NaN        0.000666   
2   SF_11_18002_1  experimental            NaN        0.000173   
3    SF_11_4018_1  experimental            NaN        0.000134   
4    SF_11_4018_2  experimental            NaN        0.000233   

   TRAV1-1;TRAJ11  TRAV1-1;TRAJ12  TRAV1-1;TRAJ13  TRAV1-1;TRAJ14  \
0        0.000096        0.000032        0.000146             NaN   
1        0.000199        0.000245        0.000337             NaN   
2        0.000041        0.000083        0.000289             NaN   
3        0.000175        0.000443        0.001328             NaN   
4        0.000272        0.000528        0.001079             NaN   

   TRAV1-1;TRAJ15  TRAV1-1;TRA